# 🦠 Group A — The Equation Model (SIR)

*You will measure this outbreak's R₀ with a deterministic model — two
sentences of common sense turned into arithmetic — and then find the
vaccination level that would have stopped it.*

### How to read this notebook

| Marker | What to do |
|---|---|
| 📖 **IDEA** | Read the explanation first |
| ✏️ **EDIT ME** | Change a value, re-run, and see what moves |
| ▶️ **RUN** | Run the code cell |
| 👀 **READ** | Inspect the result, graph, or message |
| 🧠 **BUILD IT** | A core concept turned into code — read this one closely |
| ✅ **CHECKPOINT** | Pause and answer the questions |
| 🔒 **RUN ONLY** | Infrastructure. Run it; you do not need to memorize it |
| ⭐ **OPTIONAL** | Try only after the required work is complete |

> 🖥️ **Before anything else — pick the kernel.** In the top-right corner of
> Jupyter, the kernel should read **`Python (epidemic-modeling)`**. Your site
> may name it differently — for example `Python (<site_kernel_name>)` — so if
> you don't see it, ask your project lead *before* debugging anything.
> A wrong kernel makes the setup cell fail with `ModuleNotFoundError`.

> **Never written Python before? That is expected here.** Follow the markers,
> read the “What the next cell does” notes, and connect each graph back to the
> question. If the code itself feels unfamiliar, the **master notebook at the
> repository root** opens with a 🧭 *Python survival guide* and a 🧩 *function
> map* of every tool — keep it open in another tab.

## Project path

The whole project — including the presentation and practice — fits within
**10–12 hours**.

> **Shared question** *(when does an outbreak explode, and how much
> vaccination stops it?)*
> ➜ **Group A: the SIR equations (smooth, average behavior)**
> ➜ **Shared measurement** *(this outbreak's R₀)*
> ➜ **Shared experiment** *(the vaccination threshold)*
> ➜ **Group presentation** *(equations vs coin flips — do they agree?)*

## The research question

> **When does an outbreak explode — and how much vaccination stops it?**

A disease has swept through a town of **10,000 people**. All we have is the
public-health record: how many people got sick each day, for 150 days
(`data/observed_outbreak.csv`). You are the disease detectives.

Epidemiologists describe outbreaks with three groups of people and one number:

| Symbol | Who they are |
|---|---|
| **S** — Susceptible | could still catch it |
| **I** — Infectious | sick now, and spreading it |
| **R** — Recovered | had it, now immune |

**R₀ ("R-naught")** = how many people one sick person infects, on average,
when everyone around them is susceptible. If R₀ > 1 the outbreak grows; if
R₀ < 1 it dies out. Your two jobs:

1. **Measure this outbreak's R₀** from the daily-case record.
2. **Find the vaccination level that would have prevented it** — and compare
   your answer with the famous formula for herd immunity, **v\* = 1 − 1/R₀**.

## Setup — run this first

### What the next cell does 📖

1. **Imports the toolbox.** Think of an import as: *“Python, please give me
   this toolbox.”* — `numpy` does math on whole lists of numbers at once,
   `matplotlib` draws graphs.
2. **Loads the outbreak record** from `data/observed_outbreak.csv` into an
   array called `observed` — one number per day: how many people got sick.
3. **Loads three small helper tools** (explained right above their use) and
   **prints a check** so you know everything is ready.

In [ ]:
import sys
sys.path.insert(0, "..")   # helper tools live at the repository root
import numpy as np
import matplotlib.pyplot as plt

# three small tested tools (🔒 in epidemic_helpers.py — see the master's 🧩 map):
#   align_to_threshold : shift a curve so day 0 = the day it reached 20 cases
#   rmse               : the typical difference between two curves
#   analytic_final_size: textbook prediction of the outbreak's final size
from epidemic_helpers import align_to_threshold, rmse, analytic_final_size

observed = np.loadtxt("../data/observed_outbreak.csv",
                      delimiter=",", skiprows=1)[:, 1]

N = 10_000          # people in the town
GAMMA = 0.25        # recovery rate: 1 / (average 4 days infectious)

print(f"days of records: {len(observed)}")
print(f"total people infected: {int(observed.sum()) + 5} of {N}")
print("Preflight OK — you are ready to start.")

## 👀 Meet the outbreak

### What the next cell does 📖

Two pictures of the epidemic: **left** — how many people got sick each day
(the famous “epidemic curve”); **right** — the running total. Steps: plot the
daily numbers, add them up with `np.cumsum`, plot the total, label everything.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
days = np.arange(len(observed))

axes[0].bar(days, observed, width=1.0, color="C3", alpha=0.8)
axes[0].set_xlabel("day"); axes[0].set_ylabel("new cases per day")
axes[0].set_title("The outbreak, day by day")

axes[1].plot(days, np.cumsum(observed), color="C3")
axes[1].set_xlabel("day"); axes[1].set_ylabel("total people infected")
axes[1].set_title("The running total")

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### ✅ CHECKPOINT — data

1. Around which day did the outbreak peak? Roughly how many fell sick that day?
2. What fraction of the town was eventually infected?
3. Before any model: why do outbreaks *stop on their own*, even with nobody
   vaccinated? (Hint: who is left to infect?)

### ✏️ EDIT ME — the only settings cell

Run the whole notebook once with these defaults. Then come back, change
**one** value, and re-run everything below this cell.

In [ ]:
# ✏️ PARTICIPANT EDIT AREA — change values here, nowhere else
R0_CANDIDATES = [1.5, 1.8, 2.0, 2.2, 2.4, 2.5, 2.6, 2.8, 3.0, 3.5]
VACCINATION_LIST = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.8]

print(f"{len(R0_CANDIDATES)} candidate R0 values, "
      f"{len(VACCINATION_LIST)} vaccination levels to test")

## 🧠 BUILD IT — the SIR model: tomorrow's counts from today's counts

Our model is two sentences of common sense, turned into arithmetic:

1. **Some susceptible people get infected today.** The more infectious people
   are around, the more likely each susceptible person is to meet one.
2. **Some infectious people recover today.** On average people are infectious
   for 4 days, so about a quarter (`GAMMA = 0.25`) recover each day.

### The Python, decoded 🧩

| Line | What it does in plain English |
|---|---|
| `def sir_day(...):` | Defines a mini-program: give it today's S, I, R — it returns tomorrow's. |
| `np.exp(-beta * I / N)` | The chance one susceptible person gets through the whole day *without* being infected. |
| `S * (1 - ...)` | So this many susceptible people, on average, DO get infected today. |
| `gamma * I` | This many infectious people recover today. |
| `for day in range(days):` | Repeat the daily update 150 times — that is the whole simulation. |

In [ ]:
def sir_day(S, I, R, beta, gamma, N):
    """One day of the epidemic: tomorrow's counts from today's counts."""
    new_infections = S * (1 - np.exp(-beta * I / N))   # who catches it today
    recoveries = gamma * I                              # who recovers today
    return (S - new_infections,
            I + new_infections - recoveries,
            R + recoveries,
            new_infections)

def simulate_sir(R0, vaccinated_frac=0.0, days=150, I0=5):
    """Run the model day by day; return the daily new-case curve."""
    beta = R0 * GAMMA                       # transmission rate from R0
    S = N * (1 - vaccinated_frac) - I0      # everyone not vaccinated...
    I = I0                                  # ...except the first 5 cases
    R = N * vaccinated_frac                 # vaccinated start immune
    new_cases = np.zeros(days)
    for day in range(days):
        S, I, R, new_cases[day] = sir_day(S, I, R, beta, GAMMA, N)
    return new_cases

print("SIR model defined — two functions, no magic")

### What the next cell does 📖

A first honest guess: **R₀ = 2.0**. Steps:

1. simulate an epidemic with `simulate_sir(2.0)`,
2. align both curves at the day they reached 20 total cases (🔒
   `align_to_threshold` — random outbreaks take off at random times, so we
   compare them from the same milestone),
3. draw the model on top of reality.

**Predict before you run:** will R₀ = 2.0 be too small, too big, or just right?

In [ ]:
obs_aligned = align_to_threshold(observed, 20)

guess = simulate_sir(2.0)
guess_aligned = align_to_threshold(guess, 20)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(range(len(obs_aligned)), obs_aligned, width=1.0, color="C3",
       alpha=0.5, label="observed outbreak")
ax.plot(guess_aligned, color="C0", lw=2, label="SIR model, R0 = 2.0")
ax.set_xlabel("days since the outbreak reached 20 cases")
ax.set_ylabel("new cases per day")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"difference between model and reality: {rmse(obs_aligned, guess_aligned):.1f} cases/day")

## 🧪 Measure R₀ — try every candidate, keep the best

This is how a lot of real science works: propose candidate values, simulate
each one, and keep the candidate whose prediction best matches reality.

### What the next cell does 📖

1. loops over your `R0_CANDIDATES` list (from the ✏️ cell),
2. simulates one epidemic per candidate and measures its difference from the
   observed curve (`rmse` — lower is better),
3. plots difference vs candidate — the dip marks your measurement,
4. redraws the best model on top of reality.

In [ ]:
errors = []
for R0 in R0_CANDIDATES:
    model = align_to_threshold(simulate_sir(R0), 20)
    errors.append(rmse(obs_aligned, model))

best_R0 = R0_CANDIDATES[int(np.argmin(errors))]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(R0_CANDIDATES, errors, "o-")
axes[0].axvline(best_R0, color="C3", ls=":")
axes[0].set_xlabel("candidate R0"); axes[0].set_ylabel("difference from reality")
axes[0].set_title(f"The dip is the answer: R0 = {best_R0}")

best = align_to_threshold(simulate_sir(best_R0), 20)
axes[1].bar(range(len(obs_aligned)), obs_aligned, width=1.0, color="C3",
            alpha=0.5, label="observed")
axes[1].plot(best, color="C0", lw=2, label=f"SIR, R0 = {best_R0}")
axes[1].set_xlabel("days since 20 cases"); axes[1].legend()
axes[1].set_title("Best-fitting epidemic")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Your measurement: this outbreak's R0 ≈ {best_R0}")
print(f"Herd-immunity formula predicts the threshold: 1 - 1/R0 = {1 - 1/best_R0:.2f}")

### ✅ CHECKPOINT — result

1. Write down your R₀ measurement — you will present it.
2. One sick person walks into a fully susceptible town. Using your R₀: how
   many people do they infect, on average?
3. The fit is close but not perfect — reality wiggles around the smooth
   curve. What is your model missing? (Group B's whole method is the answer.)

## 🧪 The shared experiment — how much vaccination stops it?

Vaccinating a fraction `v` of the town moves them straight from S to R on
day 0: they can neither catch nor spread it. Somewhere between v = 0 and
v = 0.8 the outbreak should collapse — that point is the **herd-immunity
threshold**, and the textbook formula says it sits at **1 − 1/R₀**.

### What the next cell does 📖

1. loops over your `VACCINATION_LIST`,
2. simulates one epidemic per vaccination level (your fitted R₀),
3. plots the outbreak's final size against vaccination, with the formula's
   prediction as a vertical dashed line.

**Predict before you run:** will the curve slope down gently, or fall off a
cliff?

In [ ]:
final_sizes = []
for v in VACCINATION_LIST:
    cases = simulate_sir(best_R0, vaccinated_frac=v)
    final_sizes.append(cases.sum())

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(VACCINATION_LIST, final_sizes, "o-", color="C2")
ax.axvline(1 - 1/best_R0, color="k", ls="--",
           label=f"herd-immunity formula: 1 - 1/R0 = {1 - 1/best_R0:.2f}")
ax.set_xlabel("fraction of the town vaccinated on day 0")
ax.set_ylabel("total people infected")
ax.set_title("The vaccination cliff")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### ✅ CHECKPOINT — conclusion

1. Where does *your* curve collapse? How close is that to 1 − 1/R₀?
2. Why is it a cliff and not a gentle slope? (What happens to each infection
   chain when a sick person's contacts are mostly immune?)
3. Group B measures the same threshold with coin flips instead of equations.
   Predict: will they get the same number?

## 📊 Variable and column reference

Use this table to make **your own extra graphs** without guessing what
names mean. Everything listed is in memory after running the notebook.

| Name | What it is | Good for plotting |
|---|---|---|
| `observed` | array, 150 days of new cases | the epidemic curve |
| `obs_aligned` | the same curve, day 0 = 20th case | comparisons with models |
| `N`, `GAMMA` | town size (10,000) and recovery rate (0.25/day) | — |
| `best_R0` | your measurement of this outbreak's R₀ | headline number |
| `R0_CANDIDATES`, `VACCINATION_LIST` | your ✏️ experiment settings | axes |
| `errors` | difference-from-reality per candidate | the fitting dip |
| `simulate_sir(R0, vaccinated_frac)` | your model — returns a daily-cases curve | any what-if |
| `final_sizes` | total infected per vaccination level | the cliff, replotted |

*Example:* `plt.plot(np.cumsum(observed))` — the outbreak's running total.

## ✅ CHECKPOINT — Group A presentation

Your talk should answer, with evidence:

- **Shared research question:** when does an outbreak explode, and how
  much vaccination stops it?
- **Group A choice:** smooth SIR equations — one deterministic curve per scenario
- **Shared measurement:** this outbreak's R₀ ≈ [your number]
- **Shared experiment:** the vaccination cliff — where it falls, vs the
  formula 1 − 1/R₀
- **Your method's special insight:** the vaccination *cliff*: a sharp threshold, exactly where
  1 − 1/R₀ says it should be
- **Conclusion:** [one or two evidence-based sentences]
- **Limitation and next question:** [e.g., everyone mixes equally here —
  no households, schools, or superspreaders; what would a network change?]

Presentation preparation and practice fit **inside** the program's
10–12 hour total. Compare R₀ and thresholds with the other group — do
equations and coin flips agree?

## ⭐ OPTIONAL — exploration

1. **A slower disease.** Set `GAMMA = 0.125` (8 days infectious) and refit.
   Does R₀ change? Does the threshold?
2. **Late vaccination.** Modify `simulate_sir` to move people from S to R at
   day 30 instead of day 0. How much does waiting cost?
3. The master notebook has the *analytic final-size equation* — check your
   simulation against 100-year-old mathematics.